# UCS420: Cognitive Computing — Assignment 7 (NLP-I)

**Google Colab submission format**

> **Important:** The assignment requires the exact dataset generated by `UCS420_LA9_Starter.ipynb`.
> Do not modify `generate_dataset()` or `process_ticket()`. Run/import the starter notebook code first,
> then run the solution cells below.


In [ ]:
# Step 0 — Imports and NLTK setup

import re
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

from nltk.tokenize import word_tokenize, RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
from nltk import pos_tag
from collections import Counter, defaultdict


## Starter notebook setup

Run the original starter notebook first and enter your roll number.

The code below expects these variables/functions from the starter notebook:

- `generate_dataset()`
- `process_ticket()`
- `TEST_LINES`

Do not replace those starter functions.


In [ ]:
# Run this cell only after the starter notebook code has been executed.

required_names = ["generate_dataset", "process_ticket"]
missing = [name for name in required_names if name not in globals()]

if missing:
    raise NameError(
        f"Missing starter-notebook definitions: {missing}. "
        "Run/copy the original UCS420_LA9_Starter.ipynb cells first."
    )

# Generate the exact assignment dataset using the starter function.
tickets = generate_dataset()

print(f"Number of generated tickets: {len(tickets)}")
print("First ticket:")
print(tickets[0])


## 1. Preprocessing

In [ ]:
# Run process_ticket() on every ticket.
processed_tickets = [process_ticket(ticket) for ticket in tickets]

print("Processed", len(processed_tickets), "tickets.")
print("\nExample processed result:")
print(processed_tickets[0])


### Helper functions for extracting fields

Because the exact return representation of the supplied starter `process_ticket()` may vary, this helper
accepts either a dictionary or a tuple/list and lets you inspect the processed object before building the table.


In [ ]:
# Inspect the first processed ticket carefully.
# This does not change process_ticket(); it only displays its returned structure.

if isinstance(processed_tickets[0], dict):
    print("Dictionary keys:", list(processed_tickets[0].keys()))
elif isinstance(processed_tickets[0], (list, tuple)):
    print("Sequence length:", len(processed_tickets[0]))
    for i, value in enumerate(processed_tickets[0]):
        print(f"[{i}] type={type(value).__name__}: {value}")
else:
    print("Return type:", type(processed_tickets[0]).__name__)
    print(processed_tickets[0])


### Recommended mapping

The assignment asks for:

- number of sentences
- number of tokens
- number of punctuation tokens
- number of stopwords
- number of unique tokens
- number of final tokens

If the starter function returns a dictionary, use the corresponding keys from the inspection above.
The following cell contains a robust extraction routine for common key names.


In [ ]:
# Flexible field extraction from process_ticket() output.
# If your starter notebook uses different dictionary key names, update only the aliases here.

def pick_field(obj, aliases, position=None):
    if isinstance(obj, dict):
        normalized = {str(k).strip().lower().replace(" ", "_"): v for k, v in obj.items()}
        for alias in aliases:
            if alias in normalized:
                return normalized[alias]
        for k, v in normalized.items():
            if any(alias in k for alias in aliases):
                return v
        raise KeyError(f"Could not find any of {aliases} in keys {list(normalized)}")
    if isinstance(obj, (list, tuple)) and position is not None:
        return obj[position]
    raise TypeError(
        "Unsupported process_ticket() return format. Inspect the object above and map the fields manually."
    )


rows = []

for i, result in enumerate(processed_tickets, start=1):
    row = {
        "ticket": i,
        "number_of_sentences": pick_field(
            result,
            ["number_of_sentences", "num_sentences", "sentences", "sentence_count"],
            position=0
        ),
        "number_of_tokens": pick_field(
            result,
            ["number_of_tokens", "num_tokens", "tokens", "token_count"],
            position=1
        ),
        "number_of_punctuation_tokens": pick_field(
            result,
            ["number_of_punctuation_tokens", "num_punctuation_tokens", "punctuation_tokens"],
            position=2
        ),
        "number_of_stopwords": pick_field(
            result,
            ["number_of_stopwords", "num_stopwords", "stopwords", "stopword_count"],
            position=3
        ),
        "number_of_unique_tokens": pick_field(
            result,
            ["number_of_unique_tokens", "num_unique_tokens", "unique_tokens"],
            position=4
        ),
        "number_of_final_tokens": pick_field(
            result,
            ["number_of_final_tokens", "num_final_tokens", "final_tokens"],
            position=5
        ),
    }
    rows.append(row)

preprocessing_df = pd.DataFrame(rows)

count_cols = [
    "number_of_sentences",
    "number_of_tokens",
    "number_of_punctuation_tokens",
    "number_of_stopwords",
    "number_of_unique_tokens",
    "number_of_final_tokens"
]

total_row = {"ticket": "TOTAL"}
for col in count_cols:
    total_row[col] = preprocessing_df[col].sum()

preprocessing_table = pd.concat(
    [preprocessing_df, pd.DataFrame([total_row])],
    ignore_index=True
)

display(preprocessing_table)


### Actual tokens for one selected ticket

The following cell uses the first ticket and displays raw tokenization details using standard NLTK
objects. This is useful for the required inspection output.


In [ ]:
selected_index = 0  # Change to any ticket index from 0 to len(tickets)-1

selected_ticket = tickets[selected_index]
selected_tokens = word_tokenize(selected_ticket)

punctuation_tokens = [
    tok for tok in selected_tokens
    if all(ch in r'''!"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~''' for ch in tok)
]

stop_words = set(stopwords.words("english"))
selected_stopwords = [tok for tok in selected_tokens if tok.lower() in stop_words]

final_tokens = [
    tok for tok in selected_tokens
    if tok not in punctuation_tokens and tok.lower() not in stop_words
]

print("RAW TICKET:\n", selected_ticket)
print("\nACTUAL TOKENS:")
print(selected_tokens)

print("\nPUNCTUATION TOKENS:")
print(punctuation_tokens)

print("\nSTOPWORDS:")
print(selected_stopwords)

print("\nFINAL TOKENS:")
print(final_tokens)


## 2. Regex extraction and replacement

In [ ]:
# Regex patterns required by the assignment.

# Words longer than 5 letters
long_words_pattern = r"\\b[A-Za-z]{6,}\\b"

# Numbers, decimals and versions such as 95.5 and 2.0.1
numbers_pattern = r"(?<![A-Za-z0-9])\\d+(?:\\.\\d+){0,}(?![A-Za-z0-9])"

# Capitalized words
capitalized_pattern = r"\\b[A-Z][a-z]+\\b"

# Words beginning with a vowel
vowel_words_pattern = r"\\b[AEIOUaeiou][A-Za-z]*\\b"

# Replacement patterns
email_pattern = r"\\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}\\b"
url_pattern = r"https?://[^\\s]+|www\\.[^\\s]+"
phone_pattern = r"(?<!\\d)(?:\\+?\\d[\\d() .-]{7,}\\d)(?!\\d)"

def regex_extract_and_replace(text):
    return {
        "long_words": re.findall(long_words_pattern, text),
        "numbers": re.findall(numbers_pattern, text),
        "capitalized_words": re.findall(capitalized_pattern, text),
        "vowel_words": re.findall(vowel_words_pattern, text),
        "email_replaced": re.sub(email_pattern, "<EMAIL>", text),
        "url_replaced": re.sub(url_pattern, "<URL>", text),
        "phone_replaced": re.sub(phone_pattern, "<PHONE>", text),
    }


In [ ]:
# Apply regex operations to every raw ticket.

regex_results = []

for i, ticket in enumerate(tickets, start=1):
    result = regex_extract_and_replace(ticket)
    result["ticket"] = i
    regex_results.append(result)

for result in regex_results:
    print("=" * 80)
    print("TICKET", result["ticket"])
    print("Long words:", result["long_words"])
    print("Numbers:", result["numbers"])
    print("Capitalized words:", result["capitalized_words"])
    print("Vowel-starting words:", result["vowel_words"])
    print("Email replacement:", result["email_replaced"])
    print("URL replacement:", result["url_replaced"])
    print("Phone replacement:", result["phone_replaced"])


### Test the patterns on `TEST_LINES`

This verifies that the same regex expressions are applied to the assignment's supplied test strings.


In [ ]:
for i, line in enumerate(TEST_LINES, start=1):
    print("=" * 80)
    print(f"TEST_LINE {i}:")
    print(line)

    print("Long words:", re.findall(long_words_pattern, line))
    print("Numbers:", re.findall(numbers_pattern, line))
    print("Capitalized words:", re.findall(capitalized_pattern, line))
    print("Vowel words:", re.findall(vowel_words_pattern, line))

    replaced = re.sub(email_pattern, "<EMAIL>", line)
    replaced = re.sub(url_pattern, "<URL>", replaced)
    replaced = re.sub(phone_pattern, "<PHONE>", replaced)

    print("Replaced:", replaced)


## 3. Custom tokenization

The tokenizer must preserve contractions, hyphenated words, decimals, versions, email addresses,
URLs and phone numbers. A single `RegexpTokenizer` pattern is used.


In [ ]:
custom_pattern = r"""https?://[^\\s]+|www\\.[^\\s]+|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}|\\+?\\d[\\d(). -]{7,}\\d|\\d+(?:\\.\\d+)+|[A-Za-z]+(?:-[A-Za-z]+)+|[A-Za-z]+(?:'[A-Za-z]+)|[A-Za-z]+|\\d+|[^\\w\\s]"""

custom_tokenizer = RegexpTokenizer(custom_pattern)

def my_tokenize(text):
    return custom_tokenizer.tokenize(text)


In [ ]:
print("Custom tokenizer on TEST_LINES")
for i, line in enumerate(TEST_LINES, start=1):
    print(f"\nTEST_LINE {i}")
    print("Custom:", my_tokenize(line))
    print("word_tokenize:", word_tokenize(line))


In [ ]:
# Run custom tokenization on the generated tickets.

custom_ticket_tokens = [my_tokenize(ticket) for ticket in tickets]

for i, (ticket, tokens) in enumerate(zip(tickets, custom_ticket_tokens), start=1):
    print(f"TICKET {i}:")
    print(tokens)
    print()


### Comparison with `word_tokenize()`

Typical differences to identify from the actual dataset:

1. **Contractions:** `isn't` is preserved as one token by this tokenizer, while NLTK's
   `word_tokenize()` commonly splits the contraction into multiple tokens.
2. **Hyphenated words:** `Wi-Fi` and `state-of-the-art` are preserved as single tokens here,
   whereas `word_tokenize()` can split punctuation/hyphen components.
3. **Special structured strings:** emails, URLs, phone numbers, decimals and version numbers
   are kept as single tokens by the custom pattern.

The next cell prints concrete differences from the supplied dataset.


In [ ]:
# Show concrete differences using the first few tickets.

for i, ticket in enumerate(tickets[:5], start=1):
    custom = my_tokenize(ticket)
    standard = word_tokenize(ticket)

    if custom != standard:
        print("=" * 80)
        print(f"TICKET {i}")
        print("Custom tokenizer:", custom)
        print("word_tokenize(): ", standard)


## 4. Stemming and lemmatization

In [ ]:
# Collect unique final tokens.

# Prefer the final tokens extracted by process_ticket() when available.
# This fallback recreates final tokens with word_tokenize() if the starter representation is unavailable.

all_final_tokens = []

for result in processed_tickets:
    try:
        final = pick_field(
            result,
            ["final_tokens", "final_token_list", "tokens_final"],
            position=5
        )
        if isinstance(final, str):
            final = word_tokenize(final)
        all_final_tokens.extend(final)
    except Exception:
        pass

if not all_final_tokens:
    for ticket in tickets:
        toks = word_tokenize(ticket)
        final = [
            tok for tok in toks
            if tok not in set(stopwords.words("english"))
            and not all(ch in r'''!"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~''' for ch in tok)
        ]
        all_final_tokens.extend(final)

unique_final_tokens = sorted(set(all_final_tokens), key=lambda x: x.lower())

print("Number of final tokens:", len(all_final_tokens))
print("Number of unique final tokens:", len(unique_final_tokens))
print("Sample:", unique_final_tokens[:30])


In [ ]:
porter = PorterStemmer()
lancaster = LancasterStemmer()
lemmatizer = WordNetLemmatizer()

# A suffix chosen using common words likely to occur in an IT-support ticket dataset.
# Example: 'connected' and 'installed' both contain the suffix 'ed'.
regexp_stemmer = RegexpStemmer(r"ed$", min=3)

def penn_to_wordnet(tag):
    if tag.startswith("J"):
        return "a"  # adjective
    if tag.startswith("V"):
        return "v"  # verb
    if tag.startswith("N"):
        return "n"  # noun
    if tag.startswith("R"):
        return "r"  # adverb
    return "n"

# POS-tag unique final tokens.
tagged = pos_tag(unique_final_tokens)

rows = []
for word, tag in tagged:
    word_lower = word.lower()
    porter_out = porter.stem(word_lower)
    lancaster_out = lancaster.stem(word_lower)
    regexp_out = regexp_stemmer.stem(word_lower)
    wn_pos = penn_to_wordnet(tag)
    lemma_out = lemmatizer.lemmatize(word_lower, pos=wn_pos)

    rows.append({
        "word": word,
        "POS": tag,
        "Porter": porter_out,
        "Lancaster": lancaster_out,
        "Regexp": regexp_out,
        "WordNet_lemma": lemma_out
    })

stemming_df = pd.DataFrame(rows)

different_df = stemming_df[
    (stemming_df["Porter"] != stemming_df["Lancaster"]) |
    (stemming_df["Porter"] != stemming_df["Regexp"]) |
    (stemming_df["Porter"] != stemming_df["WordNet_lemma"]) |
    (stemming_df["Lancaster"] != stemming_df["Regexp"]) |
    (stemming_df["Lancaster"] != stemming_df["WordNet_lemma"]) |
    (stemming_df["Regexp"] != stemming_df["WordNet_lemma"])
].reset_index(drop=True)

display(different_df)


### Regexp Stemmer justification

The suffix `ed` was selected because past-tense/past-participle words such as **connected** and
**installed** are typical examples in IT-support tickets. Removing the `ed` suffix demonstrates how
`RegexpStemmer` can be customized for a specific corpus.


## 5. Corpus statistics

In [ ]:
# Build corpus token streams.

# S1: all tokens
S1 = []
for ticket in tickets:
    S1.extend(word_tokenize(ticket))

# S2: tokens without punctuation
punctuation_set = set(r'''!"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~''')
S2 = [tok for tok in S1 if tok not in punctuation_set]

# S3: final tokens — remove punctuation and English stopwords.
stop_words = set(stopwords.words("english"))
S3 = [
    tok for tok in S2
    if tok.lower() not in stop_words
]

# S4: Porter-stemmed S3
S4 = [porter.stem(tok.lower()) for tok in S3]

def corpus_stats(tokens):
    total = len(tokens)
    unique = len(set(tokens))
    ttr = unique / total if total else 0
    return total, unique, ttr

stats = {
    "S1 (all tokens)": corpus_stats(S1),
    "S2 (without punctuation)": corpus_stats(S2),
    "S3 (final tokens)": corpus_stats(S3),
    "S4 (Porter stemmed)": corpus_stats(S4)
}

stats_df = pd.DataFrame(
    stats,
    index=["Total tokens", "Unique tokens", "TTR"]
).T

display(stats_df)


### Top 10 words of S1 and S3

In [ ]:
top10_S1 = Counter(tok.lower() for tok in S1).most_common(10)
top10_S3 = Counter(tok.lower() for tok in S3).most_common(10)

print("Top 10 words — S1")
for word, freq in top10_S1:
    print(f"{word}: {freq}")

print("\nTop 10 words — S3")
for word, freq in top10_S3:
    print(f"{word}: {freq}")


### Zipf plot: log(rank) vs log(frequency) for S3

In [ ]:
freqs = np.array(sorted(Counter(tok.lower() for tok in S3).values(), reverse=True))

ranks = np.arange(1, len(freqs) + 1)

plt.figure(figsize=(8, 5))
plt.plot(np.log(ranks), np.log(freqs), marker="o", markersize=3, linewidth=1)
plt.xlabel("log(rank)")
plt.ylabel("log(frequency)")
plt.title("Zipf-style Plot for S3")
plt.grid(True, alpha=0.3)
plt.show()


### Top 5 bigrams of S1 and S3

In [ ]:
def bigrams(tokens):
    return list(zip(tokens[:-1], tokens[1:]))

top5_bigram_S1 = Counter(bigrams([tok.lower() for tok in S1])).most_common(5)
top5_bigram_S3 = Counter(bigrams([tok.lower() for tok in S3])).most_common(5)

print("Top 5 bigrams — S1")
for bg, freq in top5_bigram_S1:
    print(f"{bg}: {freq}")

print("\nTop 5 bigrams — S3")
for bg, freq in top5_bigram_S3:
    print(f"{bg}: {freq}")


## 6. Interpretation

### (a) Why does TTR change from S1 to S4?

TTR is `unique tokens / total tokens`.

From S1 to S4, punctuation and stopwords have already been removed by the later stages, and stemming
also maps several related word forms to a common stem. Therefore the number of distinct types generally
falls more than the total token count, causing TTR to change. The exact direction and magnitude should
be taken from the statistics printed above for this generated dataset.


### (b) One difference between `word_tokenize()` and the custom tokenizer

For example, **`isn't`** is intentionally preserved as one token by the custom tokenizer. Standard
`word_tokenize()` can split the contraction into multiple tokens. This happens because the two tokenizers
use different tokenization rules.


### (c) One undesirable stemming result

A stemmer can produce a non-word or semantically coarse form. For example, Porter stemming can reduce
related terms to a stem that is not itself a normal English word. This is a limitation of stemming:
it operates primarily by suffix-removal rules rather than by understanding the word's meaning.


### (d) POS-aware lemmatization vs stemming

POS-aware lemmatization can preserve meaning more accurately because it uses the word's grammatical role.
For instance, a verb such as **`running`** can be lemmatized to **`run`** when tagged as a verb, whereas
a stemmer may return a shortened form that is not an actual dictionary word. The same principle applies
to nouns, adjectives and adverbs in the ticket corpus.


## Submission checklist

Before submitting:

1. Run the original starter notebook and enter your roll number.
2. Execute all cells in this notebook from top to bottom.
3. Verify the preprocessing table and TOTAL row.
4. Verify all regex outputs on both tickets and `TEST_LINES`.
5. Check the custom tokenizer differences against `word_tokenize()`.
6. Check the stemming/lemmatization difference table.
7. Save the Colab notebook with all output cells populated.
